# Harness evolution: deterministic reference

**What it does.** Demonstrates how changing an agent's instructions or available
context changes the code it produces, while independent tests judge the repairs.

**How it works.** Handwritten proposer and worker fixtures produce bounded
harness revisions and real source edits. Separate development, private-selection,
and held-out checks retain the measurements, citations, failures, and usage;
only an eligible selected harness can be exported. This is a deterministic
mechanism study, separate from the [live Meta-Harness adaptation](https://sentient-xyz.github.io/meta-evolve-docs/research/meta-harness/).

<!-- Kept together to preserve the complete runnable study and its section anchors. -->

The [live Meta-Harness lesson](https://sentient-xyz.github.io/meta-evolve-docs/research/meta-harness/) introduces real SDK agents in seven cells.
This reference preserves the shipped deterministic audit and recovery study.

**Status: Shipped.** This deterministic study runs bounded repairs and independent
tests. Its proposer and worker are handwritten fixtures; provider-backed adaptations
are **Experimental/live-only**.

## The concrete problem

An agent's instructions change its code. Here is the first revision in the shipped study:

| Observation | Starting harness | First revised harness |
|---|---|---|
| Planner instruction | `Make a conservative change while preserving the interface.` | `Diagnose requested behavior and repair arithmetic operations.` |
| Worker output in `math_ops.py` | `return value - 1` | `return value + 1` |
| Independent check | `increment(3) == 4` fails | The same check passes |

The fixed worker recognizes the revised instruction through an authored Python
branch. It produces a source file; the evaluator tests that file in a fresh
workspace. The instruction itself receives no score. This first change raises
development quality from **0.0 to 0.5** across two repair tasks. A later context
change, `diagnostics: false` → `true`, repairs normalization and reaches **1.0**.
These are fixture results, not evidence of general model improvement.

## Follow the two levels

```text
Change the agent's repair instructions or context settings
↓ Run the same worker on fixed repository repair tasks
↓ Independently test the code it produces
Keep the best measured settings
```

The editable artifact is the harness; the work being judged is its worker's
repair. This makes the homepage's method-improvement example concrete. The
[introductory help assistant](https://sentient-xyz.github.io/meta-evolve-docs/guides/harness-evolution/) teaches the same
connection with returned help answers. That small simulated assistant is a
different example; this study executes repair and test processes, with governed
edits and separate development, private-selection, and held-out phases.

## Setup and run

Download the [reference notebook](https://sentient-xyz.github.io/meta-evolve-docs/downloads/harness-reference.ipynb) for a
fresh **local Jupyter** session with Python 3.12+. The verified path is macOS
with working `sandbox-exec`; Linux requires Landlock ABI 3 or newer. A supported
OS alone is insufficient: the launch check below must pass. Windows is not a
target, and hosted runtimes such as Colab are not verified for this study.

In [ ]:
import platform
import sys

if sys.version_info < (3, 12) or sys.platform not in {"darwin", "linux"}:
    raise RuntimeError("Use Python 3.12+ on macOS or Linux with confinement.")
print(platform.platform())
print(sys.version.split()[0])

Use a fresh notebook environment running **Python 3.12 or newer**.
Install directly from the published documentation:

In [ ]:
%pip install https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip

If you already imported Meta-Evolve, restart the kernel after installing.
Then run the remaining cells in order.

**Archived or offline docs:** use the ZIP included with that build. Put
`meta-evolve.zip` in the notebook's working folder (`%pwd` shows it; hosted
notebooks let you upload files), then run `%pip install ./meta-evolve.zip`
instead. Installing from source may still download build tools.

Download the complete study from the same documentation build as the package,
including its evaluator, confinement, and phase controls. No credentials are needed.

In [ ]:
from hashlib import sha256
from io import BytesIO
from pathlib import Path
import subprocess
import tempfile
from urllib.request import urlopen
from zipfile import ZipFile

downloads = "https://sentient-xyz.github.io/meta-evolve-docs/downloads/"
archive = urlopen(downloads + "harness-example.zip", timeout=30).read()
study_directory = Path(tempfile.mkdtemp(prefix="meta-harness-source-"))
with ZipFile(BytesIO(archive)) as source_zip:
    source_zip.extractall(study_directory)
example = study_directory / "examples/research/harness_evolution"
run_root = Path(tempfile.mkdtemp(prefix="meta-harness-run-"))
sys.path.insert(0, str(example))
print("example archive sha256:", sha256(archive).hexdigest())
print("results:", run_root)

Before any study work, launch a real confined process. The parent can access
an existing decoy in the protected output directory. The child must read/write
its own workspace and be denied both reading and writing that decoy. A failed
launcher does not count as successful denial. This checks protected roots;
it does not claim that macOS blocks every external read.

In [ ]:
import confinement_policy

def assert_confinement(output_root):
    decoy = output_root / ".confinement-canary"
    decoy.write_text("protected")
    assert decoy.read_text() == "protected"
    script = '''from pathlib import Path
import sys
local = Path("local.txt")
local.write_text("local-ok")
assert local.read_text() == "local-ok"
for mode in ("r", "a"):
    try:
        with open(sys.argv[1], mode) as target:
            target.read() if mode == "r" else target.write("leaked")
    except PermissionError:
        continue
    raise RuntimeError("protected file was accessible: " + mode)
print("confinement: local read/write allowed; protected read/write denied")
'''
    try:
        with tempfile.TemporaryDirectory(prefix="harness-canary-") as scratch:
            prefix = confinement_policy.launcher(Path(scratch), output_root=output_root)
            checked = subprocess.run(
                [*prefix, sys.executable, "-I", "-c", script, str(decoy)],
                cwd=scratch, env={}, capture_output=True, text=True, timeout=15,
            )
        if checked.returncode:
            raise RuntimeError("Confinement launch/check failed: " + checked.stderr)
        expected = "confinement: local read/write allowed; protected read/write denied"
        if checked.stdout.strip() != expected:
            raise RuntimeError("Confinement check did not finish: " + checked.stdout)
        assert decoy.read_text() == "protected"
        print(checked.stdout.strip())
    finally:
        decoy.unlink()

assert_confinement(run_root)

If this raises, stop here. Use a local runtime where confinement can launch;
an enclosing sandbox can deny `sandbox-exec` even when it exists. Keep protection
enabled. The existing `run_flagship` owns registration, search, private
selection, held-out comparison, and export; inspect its
[complete source](#complete-supporting-source) for those declarations.

In [ ]:
from main import run_flagship
from flagship_report import report
result = run_flagship(run_root)

### Inspect the repair

Read actual retained attempts from the first two development evaluations. The fixed
grader checks `increment(3) == 4`; score 1 means its process passed. Captured source excludes hidden tests.

In [ ]:
starting, revised = result.search.trials()[:2]
for label, trial in (("starting", starting), ("revised", revised)):
    capture = next(e for e in trial.evidence if e.kind == "episode-capture")
    repair = next(e for e in capture.data["episodes"] if e["name"] == "repair-increment")
    attempt = repair["attempts"][-1]
    print(label + " instruction:", trial.artifact.value.planner.value)
    print(attempt["source"]["math_ops.py"].strip())
    print("independent check score:", attempt["metrics"]["score"])
    print("hidden test in candidate:", "_meta_evolve_test.py" in attempt["source"])
print("development:", [t.metrics["quality"] for t in result.search.trials()])
# Output:
# starting instruction: Make a conservative change while preserving the interface.
# def increment(value):
#     return value - 1
# independent check score: 0.0
# hidden test in candidate: False
# revised instruction: Diagnose requested behavior and repair arithmetic operations.
# def increment(value):
#     return value + 1
# independent check score: 1.0
# hidden test in candidate: False
# development: [0.0, 0.5, 1.0]

The full report retains phase totals and ceilings, citations, selection, comparison,
and export status. Token and provider-call counts describe simulated fixture usage;
no model was contacted or billed.

In [ ]:
print(report(result))
print("export location:", result.exported.destination if result.exported else None)

```text title="Verified fixture excerpt"
development=[0.0, 0.5, 1.0]
```

```text title="Verified fixture excerpt"
held_out_baseline=0.0
held_out_candidate=1.0
verdict=improvement
promotion_eligible=True
```

Keep `run_root` to inspect results; `run_flagship(run_root)` resumes compatible phases.
For another study, restart the kernel and rerun the cells for fresh directories.
From a checkout, the equivalent terminal route is:

```bash
uv sync --locked
uv run python examples/research/harness_evolution/main.py --output /tmp/meta-evolve-harness
```

## Component worksheet

| Question | This implementation |
|---|---|
| What changes? | Typed planner text and context-policy configuration |
| What proposes it? | A confined fixture uses development diagnosis and bounded authorized source reads; `tfd` validates and applies one cited mutation |
| What judges it? | Independent repository repair episodes measure `quality` |
| What stays fixed? | Worker, hidden tests, split membership, component declarations, confinement, and phase budgets |
| When does work stop? | Development allows two revisions, three evaluations, 100 fixture tokens, and 30 seconds; it may stop on satisfaction |
| What is retained? | Phase runs, captured repairs, source observations, citations, predictions, failures, and eligible export |

Each inner episode allows one repair attempt, two evaluations, 20 fixture tokens,
and five seconds. Each test process has a two-second timeout.

## Declare allowed changes

This study uses a `HarnessArtifact` to demonstrate its guarantees: component/interface
declarations stay immutable, only the two declared values can change, and comparison/export
preserve their identity. See the canonical [harness semantics](https://sentient-xyz.github.io/meta-evolve-docs/architecture/#programming-harnesses).

### The actual development declaration

Registration supplies the named evaluator and proposer; execution binds confinement
to the output directory and preserves registered resume. Inspect the declaration in
[`_search`](https://github.com/sentient-xyz/meta-evolve/blob/main/examples/research/harness_evolution/flagship_components.py).
The web page also displays that declaration as an extracted excerpt.



## How a cited edit becomes a candidate

Development publishes diagnosis and canonical source observations through governed channels.
The proposer cites authorized observations and records a prediction and regression risk. The
[`tfd` adapter](https://github.com/sentient-xyz/meta-evolve/blob/main/examples/research/harness_evolution/tfd.py)
validates that manifest before applying one declared mutation. The web page shows
the extracted mutation below; the linked adapter owns the complete implementation.



Invalid output, citation refusal, timeout, or infrastructure failure remains typed
and unrankable, with evidence. A failed check on a completed repair is an ordinary
measured mistake. Predictions are claims, not proof.

## Check fresh repairs

After development, private selection checks **every rankable occurrence**,
including the starting harness, and can overturn the development winner. Each
candidate gets a seed-only evaluation with 40 fixture tokens and ten seconds;
ties favor the earlier occurrence. If none is rankable, there is no selection.

The selection freezes before the unchanged baseline and private winner each
receive one matched held-out evaluation under the same limits. Private and
held-out results never feed the proposer. Inner attempts/evaluations stay inner
counts; reported token and elapsed usage roll up. Decision and phase stores
retain failures, accounting, and the selection's causal record.

## Export and adopt

Only an eligible held-out comparison exports `selected-harness`. Its planner, configuration,
and manifest are available for the user to adopt. Selection and export do not deploy
anything; promotion remains external. One held-out fixture does not establish generalization.

## Change and predict

In a scratch copy, remove the second fixture edit's required citation. Predict a
typed refusal with retained evidence rather than a low-quality score. Restore the
fixture before comparing settings; evaluator and hidden tests stay outside candidate authority.

## What the paper does

[Meta-Harness, Algorithm 1](https://arxiv.org/html/2603.28052v1#S3) evaluates an initial
population of harness programs, stores code, scores, and traces, and lets a proposer
inspect that filesystem before proposing further programs. Valid proposals enter
history; the algorithm returns a Pareto frontier while the task model stays fixed.

This fixture changes two typed surfaces under bounded Greedy search. Its audited
source reader uses neither `GitHistoryView` nor the paper's unrestricted history filesystem.
Private selection and matched final comparison add controls. This does **not** reproduce
the paper's system, unrestricted program rewriting, benchmarks, or reported gains.

## Evidence and ownership

[Transcript tests](https://github.com/sentient-xyz/meta-evolve/blob/main/tests/test_harness_readme_transcript.py)
check the complete report. Study tests also cover confinement, source-read bounds,
citation authority, private selection, and replay.
[#222](https://github.com/sentient-xyz/meta-evolve/issues/222) owns the completed
implementation; [#192](https://github.com/sentient-xyz/meta-evolve/issues/192) owns research extensions.

## Complete supporting source

Download the [complete harness example](https://sentient-xyz.github.io/meta-evolve-docs/downloads/harness-example.zip) and matching
[source package](https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip), or inspect the
[canonical example directory](https://github.com/sentient-xyz/meta-evolve/tree/main/examples/research/harness_evolution),
which owns the short excerpts above.